# S&P 500 Diversification Challenge — Solution

**Find the best diversifying pair of S&P 500 stocks.** This standalone notebook contains the challenge instructions followed by a fully commented Python solution.

> **Reproducibility:** The official eligible universe is the Wikipedia S&P 500 component list downloaded on **October 7, 2026**. The notebook must use a saved copy of that list, not Wikipedia's live page, to reproduce the official ranking. Historic Yahoo Finance prices can also be revised over time.

# S&P 500 Diversification Challenge — Find the Best Diversifying Pair

You are a portfolio analyst at an investment fund. Your task is to identify two stocks from the S&P 500 that, when combined in an equally weighted portfolio, provide the **largest and most stable diversification benefit**.

You will solve two problems: first, without any return requirement, and then with a minimum return constraint.

## Your objective

Find the eligible pair of stocks with the **highest diversification score** over **January 2, 2024 to December 31, 2025**, inclusive, using daily Adjusted Close prices adjusted for stock splits and dividends.

Each portfolio must contain two different stocks with fixed weights:

$$
w_1=w_2=0.5
$$

## Eligible stocks and data

Use the tickers in **Wikipedia's S&P 500 component stocks table, based on the snapshot downloaded on October 7, 2026**. Later changes to the list do not affect this competition.

Use the same stock universe and analysis period as in the previous Tail Risk Challenge.

The following rules apply:

- Download daily **Adjusted Close** prices and calculate daily log returns.
- A stock must have valid daily returns for at least **90% of US equity trading sessions** in the analysis period.

$$
\widetilde r_{P,t}=0.5r_{1,t}+0.5r_{2,t}
$$


- Use **63-day rolling windows** containing complete daily return observations.
- Use sample standard deviations (`ddof=1`) for volatility and for the variability of diversification gains.


## Step 1 — Calculate the diversification gain

For each eligible pair of stocks, calculate the diversification gain in each rolling window:

$$
DG_t=1-\frac{\sigma_{P,t}}{0.5\sigma_{1,t}+0.5\sigma_{2,t}}
$$

where:

- $\sigma_{P,t}$ is the volatility of the 50%-50% portfolio.
- $\sigma_{1,t}$ and $\sigma_{2,t}$ are the individual stock volatilities.

All volatilities must be calculated using daily log returns over the same 63-day window.

The denominator represents the portfolio volatility that would arise if the stocks were perfectly positively correlated.

Next, calculate:

$$
Score=\operatorname{Mean}(DG)-0.5\operatorname{SD}(DG)
$$

The score rewards a high average diversification gain and penalizes instability.

**Higher scores are better.**

Also calculate the minimum diversification gain observed across the rolling windows.

## Part A — Without a return constraint

Find the eligible pair of stocks that achieves the **highest diversification score**, regardless of its mean return.

Calculate the pair's annualized mean portfolio log return:

$$
\bar r_{P,ann}=252\operatorname{Mean}(r_{P,daily})
$$

## Part B — With a return constraint

Repeat the search, but now consider only pairs whose annualized mean log return is at least as high as that of the S&P 500 index over the same period.

The condition is:

$$
\bar r_{P,ann}\geq\bar r_{S\&P500,ann}
$$

Use `^GSPC` as the S&P 500 benchmark.

Both annualized mean log returns must be calculated using the same dates and the same methodology:

$$
\bar r_{ann}=252\operatorname{Mean}(r_{daily})
$$

Among all eligible pairs satisfying the return constraint, select the one with the **highest diversification score**.

## What to submit

For **each part**, report:

1. The two selected tickers, in alphabetical order, using the original Wikipedia symbols.
2. The annualized mean portfolio return (%).
3. The mean rolling diversification gain (%).
4. The standard deviation of the rolling diversification gain (%).
5. The minimum rolling diversification gain (%).
6. The final diversification score (%).

Plot both selected pairs' rolling diversification-gain series in the same figure.

Answer the following questions:

1. Did the return constraint change the selected pair?
2. Did the highest attainable diversification score decrease? Why?
3. Which selected pair has the more stable diversification gain?
4. Why is selecting the pair with the lowest full-sample correlation not necessarily equivalent to maximizing the diversification score?

**Important:** Briefly explain your selection method, how you aligned trading dates, how you handled missing observations, and how you calculated the rolling statistics. Brute-force search is allowed.

## Financial interpretation

The challenge illustrates that diversification depends on more than simply choosing two stocks with low correlation.

A good pair should provide a **large and stable reduction in portfolio volatility**. Imposing a minimum return requirement may also change which pairs are attractive.

The diversification score measures historical risk reduction relative to a hypothetical perfectly correlated portfolio. It does not guarantee that the same diversification benefit will persist in the future.

---
# Worked solution

The solution follows the challenge in order: load the fixed universe, download and align daily data, screen stocks, calculate rolling diversification gains for every pair, and identify the best pairs under both sets of constraints.

**Implementation choice:** A 63-day window means 63 consecutive *reference trading sessions*. Missing returns invalidate the window; dates are not compressed with `dropna()` before calculating rolling statistics. For the Part B return comparison, both annualized means use the valid common return dates of the pair. This is more consistent with the challenge instructions than the original code, which compressed missing dates.

## 1. Imports and parameters

Keep the reference period and rolling-window settings together so the calculation can be repeated consistently.

In [ ]:
from itertools import combinations
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm import tqdm

START_DATE = "2024-01-02"
END_DATE = "2026-01-01"       # yfinance excludes this date
LAST_DATE = "2025-12-31"
FIRST_PRICE_DATE = "2023-12-28"  # Provides a previous close for first-day returns
WINDOW = 63
MIN_COVERAGE = 0.90

# REQUIRED for the official challenge: CSV with a Symbol column from
# Wikipedia's S&P 500 components table, saved on October 7, 2026.
UNIVERSE_FILE = Path("sp500_components_2026-10-07.csv")

## 2. Load the fixed stock universe

The two ticker conventions are kept separately: original Wikipedia symbols for reporting and Yahoo Finance symbols for downloading prices (for example, `BRK.B` becomes `BRK-B`).

**Before running:** Put `sp500_components_2026-10-07.csv` in the notebook's working directory. The CSV must contain a `Symbol` column. If you do not have the frozen snapshot, you must obtain it from the instructor; substituting the current Wikipedia table would define a different competition.

In [ ]:
if not UNIVERSE_FILE.exists():
    raise FileNotFoundError(
        f"Missing {UNIVERSE_FILE}. The challenge requires the fixed "
        "Wikipedia S&P 500 component snapshot from October 7, 2026."
    )

wikipedia_tickers = (
    pd.read_csv(UNIVERSE_FILE)["Symbol"].dropna().astype(str).str.strip().tolist()
)
assert len(wikipedia_tickers) == len(set(wikipedia_tickers)), "Duplicate ticker symbols"

yahoo_to_wikipedia = {symbol.replace(".", "-"): symbol for symbol in wikipedia_tickers}
sp500_tickers = list(yahoo_to_wikipedia)
print(f"Stocks in the fixed universe: {len(sp500_tickers)}")
print("First symbols:", wikipedia_tickers[:10])

## 3. Download the S&P 500 benchmark and define trading dates

Use `^GSPC` to define the reference trading calendar. Prices begin just before 2024 so that the first return in the challenge period can be calculated.

In [ ]:
benchmark_prices = yf.download(
    "^GSPC",
    start=FIRST_PRICE_DATE,
    end=END_DATE,
    auto_adjust=False,
    multi_level_index=False,
    progress=False
)["Adj Close"]

sp500_returns = np.log(benchmark_prices / benchmark_prices.shift(1))
sp500_returns = sp500_returns.loc[START_DATE:LAST_DATE].dropna()
dates = sp500_returns.index

if dates.empty:
    raise ValueError("No benchmark returns were downloaded.")
print(f"Reference sessions: {len(dates)}")
print(f"First / last session: {dates.min().date()} / {dates.max().date()}")

## 4. Download stock returns and enforce the 90% coverage rule

Return observations are aligned to the **same benchmark calendar**. A missing return remains missing; it is not replaced with zero. Screening is conducted *after* alignment to all reference sessions.

In [ ]:
prices = yf.download(
    sp500_tickers,
    start=FIRST_PRICE_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)["Adj Close"]

if isinstance(prices, pd.Series):
    prices = prices.to_frame(name=sp500_tickers[0])

# Missing symbols become NaN columns and fail the coverage requirement.
prices = prices.reindex(columns=sp500_tickers)

# Calculate returns first, preserving the previous close; then align dates.
log_returns = np.log(prices / prices.shift(1)).reindex(dates)
coverage = log_returns.notna().mean()
eligible_tickers = coverage.index[coverage >= MIN_COVERAGE]
log_returns = log_returns.loc[:, eligible_tickers]

assert log_returns.index.equals(sp500_returns.index)
print(f"Eligible stocks: {log_returns.shape[1]}")
print(f"Possible pairs: {log_returns.shape[1] * (log_returns.shape[1] - 1) // 2:,}")
print(f"Missing returns after screening: {int(log_returns.isna().sum().sum())}")

## 5. Compute diversification gain and score for every pair

For each pair, the 50%-50% portfolio return is the weighted sum of its two daily log returns. `rolling(WINDOW).std(ddof=1)` calculates the sample volatility only when **all 63 reference sessions** have valid observations for both stocks.

The score is `Mean(DG) − 0.5 × SD(DG)`. The Part B benchmark is computed using exactly the dates with valid portfolio returns. `tqdm` displays the progress of the exhaustive pair search.

In [ ]:
results = []
stock_pairs = combinations(log_returns.columns, 2)
n = log_returns.shape[1]
total_pairs = n * (n - 1) // 2

for stock1, stock2 in tqdm(stock_pairs, total=total_pairs, desc="Evaluating pairs"):
    # Keep the complete reference calendar: do NOT drop missing rows here.
    pair = log_returns[[stock1, stock2]]
    r1, r2 = pair[stock1], pair[stock2]
    portfolio = 0.5 * r1 + 0.5 * r2

    # A window containing any missing return produces NaN volatility.
    sd1 = r1.rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    sd2 = r2.rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    sdp = portfolio.rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    denominator = 0.5 * sd1 + 0.5 * sd2

    dg = (1 - sdp / denominator.where(denominator > 0)).dropna()
    if len(dg) < 2:  # At least two values to calculate SD(DG)
        continue

    # The 50%-50% portfolio and benchmark must use the same valid dates.
    common_dates = portfolio.dropna().index
    if len(common_dates) == 0:
        continue

    pair_return_ann = 252 * portfolio.loc[common_dates].mean()
    benchmark_return_ann = 252 * sp500_returns.loc[common_dates].mean()
    mean_dg = dg.mean()
    sd_dg = dg.std(ddof=1)

    results.append({
        "Stock 1": stock1,
        "Stock 2": stock2,
        "Return": pair_return_ann,
        "Benchmark Return": benchmark_return_ann,
        "Mean DG": mean_dg,
        "SD DG": sd_dg,
        "Min DG": dg.min(),
        "Score": mean_dg - 0.5 * sd_dg,
        "Valid Windows": len(dg),
    })

if not results:
    raise ValueError("No pair has enough complete rolling windows.")

statistics = (
    pd.DataFrame(results)
    .sort_values("Score", ascending=False, kind="stable")
    .reset_index(drop=True)
)
display(statistics.head(10))

## 6. Part A — No return constraint

Select the highest-scoring pair from all eligible combinations.

In [ ]:
part_a = statistics.iloc[0]
print("PART A — Without return constraint")
print(f"Selected pair: {part_a['Stock 1']}, {part_a['Stock 2']}")
print(f"Annualized return: {part_a['Return']:.2%}")
print(f"Mean DG: {part_a['Mean DG']:.2%}")
print(f"SD DG: {part_a['SD DG']:.2%}")
print(f"Minimum DG: {part_a['Min DG']:.2%}")
print(f"Final score: {part_a['Score']:.2%}")

## 7. Part B — Meet or exceed the S&P 500 return

Remove combinations that do not satisfy the annualized mean-return requirement, then select the highest remaining score.

In [ ]:
statistics_b = statistics.loc[
    statistics["Return"] >= statistics["Benchmark Return"]
]
if statistics_b.empty:
    raise ValueError("No pair satisfies the benchmark return constraint.")

part_b = statistics_b.iloc[0]
print("PART B — With return constraint")
print(f"Selected pair: {part_b['Stock 1']}, {part_b['Stock 2']}")
print(f"Annualized return: {part_b['Return']:.2%}")
print(f"S&P 500 benchmark return: {part_b['Benchmark Return']:.2%}")
print(f"Mean DG: {part_b['Mean DG']:.2%}")
print(f"SD DG: {part_b['SD DG']:.2%}")
print(f"Minimum DG: {part_b['Min DG']:.2%}")
print(f"Final score: {part_b['Score']:.2%}")

## 8. Compare the solutions

The return constraint cannot increase the maximum attainable score: Part B optimizes over a subset of the Part A pairs. Both parts may nevertheless select the same pair.

In [ ]:
comparison = pd.DataFrame([part_a, part_b], index=["Part A", "Part B"])
comparison["Selected Pair"] = comparison.apply(
    lambda row: ",".join(sorted([
        yahoo_to_wikipedia[row["Stock 1"]],
        yahoo_to_wikipedia[row["Stock 2"]]
    ])),
    axis=1
)
percent_columns = ["Return", "Benchmark Return", "Mean DG", "SD DG", "Min DG", "Score"]
summary = comparison[["Selected Pair", *percent_columns]].copy()
summary[percent_columns] = summary[percent_columns] * 100
display(summary.round(3))

same_pair = comparison.loc["Part A", "Selected Pair"] == comparison.loc["Part B", "Selected Pair"]
print("Did the return constraint change the best pair?", "No" if same_pair else "Yes")

## 9. Plot both winners' rolling diversification gains

Recalculate each selected pair's rolling gain on the **same reference calendar** used in the search. If both parts select the same pair, their lines overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for label, winner in [("Part A", part_a), ("Part B", part_b)]:
    stock1, stock2 = winner["Stock 1"], winner["Stock 2"]
    pair = log_returns[[stock1, stock2]]
    portfolio = 0.5 * pair[stock1] + 0.5 * pair[stock2]

    sd1 = pair[stock1].rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    sd2 = pair[stock2].rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    sdp = portfolio.rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
    denom = 0.5 * sd1 + 0.5 * sd2
    dg = 1 - sdp / denom.where(denom > 0)

    original_pair = ",".join(sorted([yahoo_to_wikipedia[stock1], yahoo_to_wikipedia[stock2]]))
    ax.plot(dg.index, dg * 100, label=f"{label}: {original_pair}", alpha=0.85)

ax.set(title="Rolling diversification gain — Selected pairs", xlabel="Date",
       ylabel="Diversification gain (%)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 10. Export the top 20 pairs for Moodle

The quiz asks students for the two tickers, in alphabetical order, in each part. For ranking-based partial credit, list the top 20 eligible answers separately for Part A and Part B.

In [ ]:
top_a = statistics.head(20)
top_b = statistics_b.head(20)

for label, top in [("PART A", top_a), ("PART B", top_b)]:
    print(f"\n{'=' * 55}\n{label} — TOP 20 PAIRS\n{'=' * 55}")
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        pair = ",".join(sorted([
            yahoo_to_wikipedia[row["Stock 1"]],
            yahoo_to_wikipedia[row["Stock 2"]]
        ]))
        print(f"{rank:2d}. {pair:16s} Score: {row['Score']:.4%}")

## 11. Interpretation and checks

1. **Did the constraint change the winner?** Check `same_pair`. If Part A already satisfies the return requirement, the two winners coincide.
2. **Did the highest score decrease?** It cannot increase when a constraint is added; it may stay equal.
3. **Which gain is more stable?** Compare the winning pairs' `SD DG` values: smaller means more stable.
4. **Why not just choose the lowest full-sample correlation?** The score uses *rolling* volatility reductions and subtracts a penalty for variability; the correlation over the full sample alone cannot capture those features.

**Reference-data warning:** The original notebook downloaded the *current* Wikipedia table, then accidentally overwrote calendar-aligned returns with a second, unaligned calculation. This solution removes that duplicate and implements consecutive-session rolling windows. Consequently, historical scores previously reported from the old version need not match the scores produced here, even when the ticker universe is the same.